In [2]:
from pathlib import Path

OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W = 10
H = 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"

SURFACE_ALPHA = 0.28

GRID_STEP_U = 8
GRID_STEP_V = 8

GRID_GLOW_WIDTH = 1.4
GRID_CORE_WIDTH = 0.45

GRID_GLOW_ALPHA = 0.55
GRID_CORE_ALPHA = 0.95

COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

# Lemniscate Torus

In [3]:
# Lemniscate Torus — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/lemniscate_torus_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "lemniscate_torus_v1_grid"


# -----------------------------------------------------------------------------
# Lemniscate of Bernoulli tube
#
# r² = a² cos(2t)
#
# Then thickened into a tube.
# -----------------------------------------------------------------------------

n_path = 600
n_tube = 48

t_path = np.linspace(0, 2 * np.pi, n_path)
theta = np.linspace(0, 2 * np.pi, n_tube)

a = 1.0

r2 = np.cos(2 * t_path)
r2 = np.where(r2 > 0.0, r2, 0.0)

r = a * np.sqrt(r2)

cx = r * np.cos(t_path)
cy = r * np.sin(t_path)

# gentle vertical modulation
cz = 0.45 * np.sin(4 * t_path)

C = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube frame
# -----------------------------------------------------------------------------

dC = np.gradient(C, axis=0)

tangent = dC / np.linalg.norm(
    dC,
    axis=1,
    keepdims=True,
)

up = np.array([0.0, 0.0, 1.0])

normal = np.cross(
    tangent,
    up,
)

bad = np.linalg.norm(normal, axis=1) < 1e-6

normal[bad] = np.cross(
    tangent[bad],
    np.array([0.0, 1.0, 0.0]),
)

normal = normal / np.linalg.norm(
    normal,
    axis=1,
    keepdims=True,
)

binormal = np.cross(
    tangent,
    normal,
)

binormal = binormal / np.linalg.norm(
    binormal,
    axis=1,
    keepdims=True,
)


tube_r = 0.09

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):

    for j in range(n_tube):

        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


# -----------------------------------------------------------------------------
# Scale
# -----------------------------------------------------------------------------

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.9
Y = Y / scale * 3.9
Z = Z / scale * 3.9


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(
    111,
    projection="3d",
)

ax.set_facecolor(BG)


def setup_axes() -> None:

    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))

    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):

    hex_color = hex_color.lstrip("#")

    return tuple(
        int(hex_color[i:i + 2], 16) / 255
        for i in (0, 2, 4)
    )


base_rgb = np.array(
    hex_to_rgb(COL)
)


def build_colors(pulse: float):

    zn = (
        Z - Z.min()
    ) / (
        Z.max() - Z.min() + 1e-9
    )

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))

    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(
        colors,
        0,
        1,
    )


def draw_visible_grid(
    pulse: float,
) -> None:

    glow_alpha = (
        GRID_GLOW_ALPHA
        * (0.78 + 0.22 * pulse)
    )

    core_alpha = (
        GRID_CORE_ALPHA
        * (0.86 + 0.14 * pulse)
    )

    for i in range(
        0,
        X.shape[0],
        GRID_STEP_U,
    ):

        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(
        0,
        X.shape[1],
        GRID_STEP_V,
    ):

        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []

tau = 2 * np.pi

for frame_idx in range(FRAMES):

    t = frame_idx / FRAMES

    pulse = (
        0.5
        + 0.5 * np.sin(
            tau * 4 * t
        )
    )

    ax.clear()

    ax.set_facecolor(BG)

    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(
            pulse
        ),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(
        pulse
    )

    fig.canvas.draw()

    width, height = (
        fig.canvas.get_width_height()
    )

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(
        height,
        width,
        4,
    )

    frames.append(
        frame.copy()
    )

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(
    f"Size: {out_file.stat().st_size / 1024:.1f} KB"
)

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/lemniscate_torus_v1_grid.webm
Frames: 192
Size: 1238.7 KB


[out#0/webm @ 0x11be04180] video:1237KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.149458%
frame=  192 fps= 57 q=32.0 Lsize=    1239KiB time=00:00:08.00 bitrate=1268.5kbits/s speed=2.37x    


# Hypotrochoid Tube

In [5]:
# Hypotrochoid Tube — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/hypotrochoid_tube_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "hypotrochoid_tube_v1_grid"


# -----------------------------------------------------------------------------
# Hypotrochoid tube geometry
#
# 2D hypotrochoid:
# x = (R - r) cos(t) + d cos((R - r) / r * t)
# y = (R - r) sin(t) - d sin((R - r) / r * t)
#
# Then lifted into 3D and thickened into a tube.
# -----------------------------------------------------------------------------

n_path = 720
n_tube = 48

t_path = np.linspace(0, 2 * np.pi, n_path)
theta = np.linspace(0, 2 * np.pi, n_tube)

R_MAJOR = 8.0
R_MINOR = 5.0
D = 5.8

k = (R_MAJOR - R_MINOR) / R_MINOR

cx = (R_MAJOR - R_MINOR) * np.cos(t_path) + D * np.cos(k * t_path)
cy = (R_MAJOR - R_MINOR) * np.sin(t_path) - D * np.sin(k * t_path)
cz = 0.55 * np.sin(6 * t_path)

C = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube frame
# -----------------------------------------------------------------------------

dC = np.gradient(C, axis=0)
tangent = dC / np.linalg.norm(dC, axis=1, keepdims=True)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / np.linalg.norm(normal, axis=1, keepdims=True)

binormal = np.cross(tangent, normal)
binormal = binormal / np.linalg.norm(binormal, axis=1, keepdims=True)

tube_r = 0.12

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


# -----------------------------------------------------------------------------
# Scale
# -----------------------------------------------------------------------------

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.9
Y = Y / scale * 3.9
Z = Z / scale * 3.9


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/hypotrochoid_tube_v1_grid.webm
Frames: 192
Size: 288.9 KB


# Epicycloid Tube

In [6]:
# Epicycloid Tube — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/epicycloid_tube_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "epicycloid_tube_v1_grid"


# -----------------------------------------------------------------------------
# Epicycloid tube geometry
#
# 2D epicycloid:
# x = (R + r) cos(t) - d cos((R + r) / r * t)
# y = (R + r) sin(t) - d sin((R + r) / r * t)
#
# Then lifted into 3D and thickened into a tube.
# -----------------------------------------------------------------------------

n_path = 720
n_tube = 48

t_path = np.linspace(0, 2 * np.pi, n_path)
theta = np.linspace(0, 2 * np.pi, n_tube)

R_MAJOR = 4.0
R_MINOR = 1.0
D = 1.0

k = (R_MAJOR + R_MINOR) / R_MINOR

cx = (R_MAJOR + R_MINOR) * np.cos(t_path) - D * np.cos(k * t_path)
cy = (R_MAJOR + R_MINOR) * np.sin(t_path) - D * np.sin(k * t_path)
cz = 0.55 * np.sin(5 * t_path)

C = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube frame
# -----------------------------------------------------------------------------

dC = np.gradient(C, axis=0)
tangent = dC / np.linalg.norm(dC, axis=1, keepdims=True)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / np.linalg.norm(normal, axis=1, keepdims=True)

binormal = np.cross(tangent, normal)
binormal = binormal / np.linalg.norm(binormal, axis=1, keepdims=True)

tube_r = 0.13

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


# -----------------------------------------------------------------------------
# Scale
# -----------------------------------------------------------------------------

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.9
Y = Y / scale * 3.9
Z = Z / scale * 3.9


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/epicycloid_tube_v1_grid.webm
Frames: 192
Size: 858.5 KB


[out#0/webm @ 0x11c623000] video:857KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.215673%
frame=  192 fps= 72 q=32.0 Lsize=     859KiB time=00:00:08.00 bitrate= 879.1kbits/s speed=2.99x    


# Borromean Rings

In [7]:
# Borromean Rings — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/borromean_rings_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "borromean_rings_v1_grid"


# -----------------------------------------------------------------------------
# Borromean-like rings
# Three mutually interlocked rings arranged orthogonally.
# -----------------------------------------------------------------------------

n_path = 360
n_tube = 44

t = np.linspace(0, 2 * np.pi, n_path)
theta = np.linspace(0, 2 * np.pi, n_tube)

RING_R = 1.35
TUBE_R = 0.12
OFFSET = 0.62


def make_tube(centerline: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    dC = np.gradient(centerline, axis=0)
    tangent = dC / np.linalg.norm(dC, axis=1, keepdims=True)

    up = np.array([0.0, 0.0, 1.0])
    normal = np.cross(tangent, up)

    bad = np.linalg.norm(normal, axis=1) < 1e-6
    normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

    normal = normal / np.linalg.norm(normal, axis=1, keepdims=True)

    binormal = np.cross(tangent, normal)
    binormal = binormal / np.linalg.norm(binormal, axis=1, keepdims=True)

    X = np.zeros((n_tube, n_path))
    Y = np.zeros((n_tube, n_path))
    Z = np.zeros((n_tube, n_path))

    for i in range(n_path):
        for j in range(n_tube):
            offset = (
                TUBE_R * np.cos(theta[j]) * normal[i]
                + TUBE_R * np.sin(theta[j]) * binormal[i]
            )

            p = centerline[i] + offset

            X[j, i] = p[0]
            Y[j, i] = p[1]
            Z[j, i] = p[2]

    return X, Y, Z


# Ring 1: XY plane
C1 = np.vstack([
    RING_R * np.cos(t),
    RING_R * np.sin(t),
    OFFSET * np.sin(2 * t),
]).T

# Ring 2: YZ plane
C2 = np.vstack([
    OFFSET * np.sin(2 * t + 2 * np.pi / 3),
    RING_R * np.cos(t),
    RING_R * np.sin(t),
]).T

# Ring 3: ZX plane
C3 = np.vstack([
    RING_R * np.sin(t),
    OFFSET * np.sin(2 * t + 4 * np.pi / 3),
    RING_R * np.cos(t),
]).T


rings = [
    make_tube(C1),
    make_tube(C2),
    make_tube(C3),
]

all_values = np.concatenate([
    arr.reshape(-1)
    for ring in rings
    for arr in ring
])

scale = np.max(np.abs(all_values))

scaled_rings = []

for X, Y, Z in rings:
    scaled_rings.append((
        X / scale * 3.45,
        Y / scale * 3.45,
        Z / scale * 3.45,
    ))


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(Z: np.ndarray, pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_grid(X: np.ndarray, Y: np.ndarray, Z: np.ndarray, pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL_GRID_U,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL_GRID_V,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    tt = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * tt)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * tt),
        azim=360 * tt,
    )

    for X, Y, Z in scaled_rings:
        ax.plot_surface(
            X, Y, Z,
            rstride=1,
            cstride=1,
            facecolors=build_colors(Z, pulse),
            linewidth=0,
            antialiased=True,
            shade=False,
            alpha=SURFACE_ALPHA,
        )
        draw_grid(X, Y, Z, pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/borromean_rings_v1_grid.webm
Frames: 192
Size: 3776.8 KB


[out#0/webm @ 0x158725d80] video:3775KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.051430%
frame=  192 fps= 32 q=32.0 Lsize=    3777KiB time=00:00:08.00 bitrate=3867.4kbits/s speed=1.33x    


# Viviani Curve Tube


In [8]:
# Viviani Curve Tube — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/viviani_curve_tube_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "viviani_curve_tube_v1_grid"


# -----------------------------------------------------------------------------
# Viviani curve tube
#
# Curve:
# x = a(1 + cos t)
# y = a sin t
# z = 2a sin(t/2)
# -----------------------------------------------------------------------------

n_path = 520
n_tube = 44

t = np.linspace(0, 2 * np.pi, n_path)
theta = np.linspace(0, 2 * np.pi, n_tube)

a = 1.0

cx = a * (1 + np.cos(t))
cy = a * np.sin(t)
cz = 2 * a * np.sin(t / 2)

# Center the curve
cx -= cx.mean()
cy -= cy.mean()
cz -= cz.mean()

C = np.vstack([cx, cy, cz]).T


def make_tube(centerline: np.ndarray, tube_r: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    dC = np.gradient(centerline, axis=0)
    tangent = dC / np.linalg.norm(dC, axis=1, keepdims=True)

    up = np.array([0.0, 0.0, 1.0])
    normal = np.cross(tangent, up)

    bad = np.linalg.norm(normal, axis=1) < 1e-6
    normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

    normal = normal / np.linalg.norm(normal, axis=1, keepdims=True)

    binormal = np.cross(tangent, normal)
    binormal = binormal / np.linalg.norm(binormal, axis=1, keepdims=True)

    X = np.zeros((n_tube, n_path))
    Y = np.zeros((n_tube, n_path))
    Z = np.zeros((n_tube, n_path))

    for i in range(n_path):
        for j in range(n_tube):
            offset = (
                tube_r * np.cos(theta[j]) * normal[i]
                + tube_r * np.sin(theta[j]) * binormal[i]
            )

            p = centerline[i] + offset

            X[j, i] = p[0]
            Y[j, i] = p[1]
            Z[j, i] = p[2]

    return X, Y, Z


X, Y, Z = make_tube(C, tube_r=0.09)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.75
Y = Y / scale * 3.75
Z = Z / scale * 3.75


# -----------------------------------------------------------------------------
# Reference sphere
# -----------------------------------------------------------------------------

su = np.linspace(0, 2 * np.pi, 90)
sv = np.linspace(0, np.pi, 50)

SU, SV = np.meshgrid(su, sv)

SPHERE_R = 3.05

SX = SPHERE_R * np.sin(SV) * np.cos(SU)
SY = SPHERE_R * np.sin(SV) * np.sin(SU)
SZ = SPHERE_R * np.cos(SV)


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(z_values: np.ndarray, pulse: float):
    zn = (z_values - z_values.min()) / (z_values.max() - z_values.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*z_values.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_grid(x_values, y_values, z_values, pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, x_values.shape[0], GRID_STEP_U):
        ax.plot(x_values[i, :], y_values[i, :], z_values[i, :],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(x_values[i, :], y_values[i, :], z_values[i, :],
                color=COL_GRID_U, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, x_values.shape[1], GRID_STEP_V):
        ax.plot(x_values[:, j], y_values[:, j], z_values[:, j],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(x_values[:, j], y_values[:, j], z_values[:, j],
                color=COL_GRID_V, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    tt = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * tt)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * tt),
        azim=360 * tt,
    )

    # faint reference sphere
    ax.plot_wireframe(
        SX, SY, SZ,
        rstride=8,
        cstride=8,
        color=COL,
        linewidth=0.25,
        alpha=0.08,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(Z, pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(X, Y, Z, pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/viviani_curve_tube_v1_grid.webm
Frames: 192
Size: 1295.5 KB


[out#0/webm @ 0x14ae11890] video:1294KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.142897%
frame=  192 fps= 53 q=32.0 Lsize=    1296KiB time=00:00:08.00 bitrate=1326.6kbits/s speed=2.22x    


# Spherical Harmonic Orbital

In [9]:
# Spherical Harmonic Orbital — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/spherical_harmonic_orbital_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "spherical_harmonic_orbital_v1_grid"


theta = np.linspace(0.0, 2 * np.pi, 280)
phi = np.linspace(0.0, np.pi, 190)

TH, PH = np.meshgrid(theta, phi)

A = 0.62
L = 6
M = 5

R = 1.0 + A * np.sin(L * PH) * np.cos(M * TH)

X = R * np.sin(PH) * np.cos(TH)
Y = R * np.sin(PH) * np.sin(TH)
Z = R * np.cos(PH)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)
    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    rn = (R - R.min()) / (R.max() - R.min() + 1e-9)

    brightness = 0.16 + 0.95 * rn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*R.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL_GRID_U,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL_GRID_V,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/spherical_harmonic_orbital_v1_grid.webm
Frames: 192
Size: 4428.3 KB


[out#0/webm @ 0x13d709640] video:4426KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.045648%
frame=  192 fps= 24 q=32.0 Lsize=    4428KiB time=00:00:08.00 bitrate=4534.6kbits/s speed=1.02x    


$$ r(\theta,\phi)=1+A\sin(l\phi)\cos(m\theta) $$

# Atomic Orbital Lobes

In [10]:
# Atomic Orbital Lobes — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/atomic_orbital_lobes_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "atomic_orbital_lobes_v1_grid"


# -----------------------------------------------------------------------------
# Atomic-orbital-like lobed surface
#
# r = |sin(L φ) cos(M θ)|^P
# -----------------------------------------------------------------------------

theta = np.linspace(0.0, 2 * np.pi, 280)
phi = np.linspace(0.0, np.pi, 190)

TH, PH = np.meshgrid(theta, phi)

L = 4
M = 3
P = 0.55

R = np.abs(np.sin(L * PH) * np.cos(M * TH)) ** P
R = 0.35 + 1.05 * R

X = R * np.sin(PH) * np.cos(TH)
Y = R * np.sin(PH) * np.sin(TH)
Z = R * np.cos(PH)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    rn = (R - R.min()) / (R.max() - R.min() + 1e-9)

    brightness = 0.16 + 0.95 * rn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*R.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL_GRID_U,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL_GRID_V,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/atomic_orbital_lobes_v1_grid.webm
Frames: 192
Size: 4484.0 KB


[out#0/webm @ 0x150611b10] video:4482KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.045037%
frame=  192 fps= 25 q=32.0 Lsize=    4484KiB time=00:00:08.00 bitrate=4591.6kbits/s speed=1.03x    


# Parametric Flower Crystal

In [11]:
# Parametric Flower Crystal — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/parametric_flower_crystal_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "parametric_flower_crystal_v1_grid"


theta = np.linspace(0.0, 2 * np.pi, 300)
phi = np.linspace(0.0, np.pi, 210)

TH, PH = np.meshgrid(theta, phi)

R = (
    1.0
    + 0.34 * np.cos(8 * TH) * np.sin(5 * PH)
    + 0.22 * np.sin(13 * TH) * np.sin(3 * PH)
    + 0.14 * np.cos(4 * TH + 2 * PH)
)

X = R * np.sin(PH) * np.cos(TH)
Y = R * np.sin(PH) * np.sin(TH)
Z = R * np.cos(PH)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    rn = (R - R.min()) / (R.max() - R.min() + 1e-9)

    brightness = 0.16 + 0.95 * rn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*R.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL_GRID_U,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL_GRID_V,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/parametric_flower_crystal_v1_grid.webm
Frames: 192
Size: 4123.0 KB


[out#0/webm @ 0x137623470] video:4121KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.047749%
frame=  192 fps= 25 q=32.0 Lsize=    4123KiB time=00:00:08.00 bitrate=4222.0kbits/s speed=1.05x    


# Torus Harmonic Field

In [12]:
# Torus Harmonic Field — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/torus_harmonic_field_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "torus_harmonic_field_v1_grid"


# -----------------------------------------------------------------------------
# Torus harmonic field geometry
#
# Base torus with radial modulation:
# R(u, v) = R0 + A1 cos(k1 u) sin(m1 v) + A2 sin(k2 u + m2 v)
# -----------------------------------------------------------------------------

u = np.linspace(0.0, 2 * np.pi, 280)
v = np.linspace(0.0, 2 * np.pi, 180)

U, V = np.meshgrid(u, v)

R0 = 2.0
r0 = 0.62

A1 = 0.22
A2 = 0.12

K1 = 8
M1 = 5

K2 = 13
M2 = 3

major_mod = R0 + A1 * np.cos(K1 * U) * np.sin(M1 * V)
minor_mod = r0 + A2 * np.sin(K2 * U + M2 * V)

X = (major_mod + minor_mod * np.cos(V)) * np.cos(U)
Y = (major_mod + minor_mod * np.cos(V)) * np.sin(U)
Z = minor_mod * np.sin(V)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.9
Y = Y / scale * 3.9
Z = Z / scale * 3.9


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    field = major_mod + minor_mod

    fn = (field - field.min()) / (field.max() - field.min() + 1e-9)

    brightness = 0.16 + 0.95 * fn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*field.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=27 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/torus_harmonic_field_v1_grid.webm
Frames: 192
Size: 3690.4 KB


[out#0/webm @ 0x131604e90] video:3688KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.052608%
frame=  192 fps= 36 q=32.0 Lsize=    3690KiB time=00:00:08.00 bitrate=3779.0kbits/s speed=1.52x    


# Torus Harmonic Field

In [13]:
# Torus Harmonic Knot Field — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/torus_harmonic_knot_field_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "torus_harmonic_knot_field_v1_grid"


# -----------------------------------------------------------------------------
# Torus harmonic knot field
#
# Base torus with knot-direction modulation:
# wave = sin(P*u + Q*v)
# -----------------------------------------------------------------------------

u = np.linspace(0.0, 2 * np.pi, 300)
v = np.linspace(0.0, 2 * np.pi, 190)

U, V = np.meshgrid(u, v)

R0 = 2.0
r0 = 0.62

P = 5
Q = 3

A1 = 0.22
A2 = 0.10

wave = np.sin(P * U + Q * V)
wave2 = np.cos(2 * P * U - Q * V)

major_mod = R0 + A1 * wave
minor_mod = r0 + A2 * wave2

X = (major_mod + minor_mod * np.cos(V)) * np.cos(U)
Y = (major_mod + minor_mod * np.cos(V)) * np.sin(U)
Z = minor_mod * np.sin(V)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.9
Y = Y / scale * 3.9
Z = Z / scale * 3.9


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    field = wave + 0.5 * wave2

    fn = (field - field.min()) / (field.max() - field.min() + 1e-9)

    brightness = 0.16 + 0.95 * fn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*field.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=27 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/torus_harmonic_knot_field_v1_grid.webm
Frames: 192
Size: 3718.8 KB


[out#0/webm @ 0x15be268c0] video:3717KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.052233%
frame=  192 fps= 39 q=32.0 Lsize=    3719KiB time=00:00:08.00 bitrate=3808.0kbits/s speed=1.64x    


Идея: тороидальная поверхность, но деформация идёт не просто по сетке u,v, а вдоль узлового направления:

$$ \sin(pu + qv) $$

Получается что-то между:

* резонансной катушкой;
* тороидальным полем;
* топологическим реактором.

# Spherical Clifford Spiral

In [14]:
# Spherical Clifford Spiral — rotating mathematical sculpture
# Output: media-site/animations/Math/spherical_clifford_spiral_v1.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "spherical_clifford_spiral_v1"


n = 9000
t_curve = np.linspace(0, 2 * np.pi, n)

P = 17
Q = 29

theta = P * t_curve
phi = np.arccos(np.sin(Q * t_curve) * 0.82)

R = 1.0 + 0.08 * np.sin(46 * t_curve)

X = R * np.sin(phi) * np.cos(theta)
Y = R * np.sin(phi) * np.sin(theta)
Z = R * np.cos(phi)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.55
Y = Y / scale * 3.55
Z = Z / scale * 3.55


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)
    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


# faint reference sphere
su = np.linspace(0, 2 * np.pi, 90)
sv = np.linspace(0, np.pi, 50)
SU, SV = np.meshgrid(su, sv)

SPHERE_R = 3.2
SX = SPHERE_R * np.sin(SV) * np.cos(SU)
SY = SPHERE_R * np.sin(SV) * np.sin(SU)
SZ = SPHERE_R * np.cos(SV)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_wireframe(
        SX, SY, SZ,
        rstride=8,
        cstride=8,
        color=COL,
        linewidth=0.25,
        alpha=0.06,
    )

    ax.plot(
        X, Y, Z,
        color="#9ffcff",
        linewidth=4.2,
        alpha=0.08 + 0.08 * pulse,
    )

    ax.plot(
        X, Y, Z,
        color=COL,
        linewidth=1.05,
        alpha=0.92,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/spherical_clifford_spiral_v1.webm
Frames: 192
Size: 6200.7 KB


[out#0/webm @ 0x1536215b0] video:6199KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.032848%
frame=  192 fps= 33 q=32.0 Lsize=    6201KiB time=00:00:08.00 bitrate=6349.5kbits/s speed=1.36x    


# Mandelbulb Lite

In [15]:
# Mandelbulb Lite — rotating fractal point sculpture
# Output: media-site/animations/Math/mandelbulb_lite_v1.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "mandelbulb_lite_v1"


# -----------------------------------------------------------------------------
# Mandelbulb-lite point cloud
#
# Escape-time approximation:
# z -> z^POWER + c
# -----------------------------------------------------------------------------

N = 72
POWER = 8
MAX_ITER = 18
BAILOUT = 2.0

SPACE_LIMIT = 1.35

xs = np.linspace(-SPACE_LIMIT, SPACE_LIMIT, N)
ys = np.linspace(-SPACE_LIMIT, SPACE_LIMIT, N)
zs = np.linspace(-SPACE_LIMIT, SPACE_LIMIT, N)

points = []

for x in xs:
    for y in ys:
        for z in zs:
            cx, cy, cz = x, y, z
            zx, zy, zz = x, y, z

            escaped_at = MAX_ITER

            for i in range(MAX_ITER):
                r = np.sqrt(zx * zx + zy * zy + zz * zz)

                if r > BAILOUT:
                    escaped_at = i
                    break

                theta = np.arctan2(np.sqrt(zx * zx + zy * zy), zz)
                phi = np.arctan2(zy, zx)

                rn = r ** POWER
                theta_n = theta * POWER
                phi_n = phi * POWER

                zx = rn * np.sin(theta_n) * np.cos(phi_n) + cx
                zy = rn * np.sin(theta_n) * np.sin(phi_n) + cy
                zz = rn * np.cos(theta_n) + cz

            # Keep near-boundary points only.
            if 6 <= escaped_at <= MAX_ITER - 1:
                points.append((x, y, z, escaped_at))

points = np.array(points, dtype=float)

PX = points[:, 0]
PY = points[:, 1]
PZ = points[:, 2]
PI = points[:, 3]

scale = np.max(np.abs([PX, PY, PZ]))

PX = PX / scale * 3.4
PY = PY / scale * 3.4
PZ = PZ / scale * 3.4

intensity = (PI - PI.min()) / (PI.max() - PI.min() + 1e-9)


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=26 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    # Glow layer
    ax.scatter(
        PX,
        PY,
        PZ,
        s=7.0,
        c=COL,
        alpha=0.05 + 0.05 * pulse,
        depthshade=False,
    )

    # Core layer
    ax.scatter(
        PX,
        PY,
        PZ,
        s=1.2 + 2.2 * intensity,
        c=COL,
        alpha=0.32 + 0.28 * intensity,
        depthshade=False,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Points: {len(points)}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/mandelbulb_lite_v1.webm
Points: 6048
Frames: 192
Size: 7740.3 KB


[out#0/webm @ 0x14761ef20] video:7738KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.026312%
frame=  192 fps= 25 q=32.0 Lsize=    7740KiB time=00:00:08.00 bitrate=7926.1kbits/s speed=1.04x    


# Quaternion Julia Lite

In [16]:
# Quaternion Julia Lite — rotating fractal point sculpture
# Output: media-site/animations/Math/quaternion_julia_lite_v1.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "quaternion_julia_lite_v1"


# -----------------------------------------------------------------------------
# Quaternion Julia-lite point cloud
#
# q -> q^2 + c
# We visualize a 3D slice with w = 0.
# -----------------------------------------------------------------------------

N = 76
MAX_ITER = 22
BAILOUT = 4.0
SPACE_LIMIT = 1.45

C = np.array([-0.18, 0.72, 0.18, -0.08], dtype=float)

xs = np.linspace(-SPACE_LIMIT, SPACE_LIMIT, N)
ys = np.linspace(-SPACE_LIMIT, SPACE_LIMIT, N)
zs = np.linspace(-SPACE_LIMIT, SPACE_LIMIT, N)

points = []


def quat_square(q: np.ndarray) -> np.ndarray:
    x, y, z, w = q

    return np.array(
        [
            x * x - y * y - z * z - w * w,
            2 * x * y,
            2 * x * z,
            2 * x * w,
        ],
        dtype=float,
    )


for x in xs:
    for y in ys:
        for z in zs:
            q = np.array([x, y, z, 0.0], dtype=float)

            escaped_at = MAX_ITER

            for i in range(MAX_ITER):
                q = quat_square(q) + C

                if np.dot(q, q) > BAILOUT:
                    escaped_at = i
                    break

            if 7 <= escaped_at <= MAX_ITER - 1:
                points.append((x, y, z, escaped_at))

points = np.array(points, dtype=float)

PX = points[:, 0]
PY = points[:, 1]
PZ = points[:, 2]
PI = points[:, 3]

scale = np.max(np.abs([PX, PY, PZ]))

PX = PX / scale * 3.4
PY = PY / scale * 3.4
PZ = PZ / scale * 3.4

intensity = (PI - PI.min()) / (PI.max() - PI.min() + 1e-9)


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=26 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.scatter(
        PX,
        PY,
        PZ,
        s=7.0,
        c=COL,
        alpha=0.045 + 0.055 * pulse,
        depthshade=False,
    )

    ax.scatter(
        PX,
        PY,
        PZ,
        s=1.1 + 2.5 * intensity,
        c=COL,
        alpha=0.28 + 0.34 * intensity,
        depthshade=False,
    )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Points: {len(points)}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/quaternion_julia_lite_v1.webm
Points: 19034
Frames: 192
Size: 5219.7 KB


[out#0/webm @ 0x15a807670] video:5218KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.038631%
frame=  192 fps= 28 q=32.0 Lsize=    5220KiB time=00:00:08.00 bitrate=5345.0kbits/s speed=1.16x    


# Aizawa Attractor Tube

In [18]:
# Aizawa Attractor Tube — rotating mathematical sculpture
# Output: media-site/animations/Math/aizawa_attractor_tube_v1.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "aizawa_attractor_tube_v1"


# -----------------------------------------------------------------------------
# Aizawa attractor
# -----------------------------------------------------------------------------

a = 0.95
b = 0.7
c = 0.6
d = 3.5
e = 0.25
f = 0.1

dt = 0.01
steps = 24000

x = np.zeros(steps)
y = np.zeros(steps)
z = np.zeros(steps)

x[0] = 0.1
y[0] = 0.0
z[0] = 0.0

for i in range(steps - 1):

    dx = (
        (z[i] - b) * x[i]
        - d * y[i]
    )

    dy = (
        d * x[i]
        + (z[i] - b) * y[i]
    )

    dz = (
        c
        + a * z[i]
        - z[i]**3 / 3
        - (x[i]**2 + y[i]**2)
        * (1 + e * z[i])
        + f * z[i] * x[i]**3
    )

    x[i + 1] = x[i] + dt * dx
    y[i + 1] = y[i] + dt * dy
    z[i + 1] = z[i] + dt * dz


# remove transient

skip = 4000

cx = x[skip:]
cy = y[skip:]
cz = z[skip:]

# decimate slightly

cx = cx[::2]
cy = cy[::2]
cz = cz[::2]

C = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube construction
# -----------------------------------------------------------------------------

n_path = len(C)
n_tube = 36

theta = np.linspace(
    0,
    2 * np.pi,
    n_tube,
)

dC = np.gradient(
    C,
    axis=0,
)

tangent = dC / np.linalg.norm(
    dC,
    axis=1,
    keepdims=True,
)

up = np.array([0.0, 0.0, 1.0])

normal = np.cross(
    tangent,
    up,
)

bad = (
    np.linalg.norm(
        normal,
        axis=1,
    ) < 1e-6
)

normal[bad] = np.cross(
    tangent[bad],
    np.array([0.0, 1.0, 0.0]),
)

normal = normal / np.linalg.norm(
    normal,
    axis=1,
    keepdims=True,
)

binormal = np.cross(
    tangent,
    normal,
)

binormal = binormal / np.linalg.norm(
    binormal,
    axis=1,
    keepdims=True,
)

tube_r = 0.05

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):

    for j in range(n_tube):

        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


# -----------------------------------------------------------------------------
# Scale
# -----------------------------------------------------------------------------

scale = np.max(
    np.abs([X, Y, Z])
)

X = X / scale * 4.6
Y = Y / scale * 4.6
Z = Z / scale * 4.6


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(
    figsize=(W, H),
    dpi=DPI,
)

fig.patch.set_facecolor(BG)

ax = fig.add_subplot(
    111,
    projection="3d",
)

ax.set_facecolor(BG)


def setup_axes():

    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))

    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color):

    hex_color = hex_color.lstrip("#")

    return tuple(
        int(hex_color[i:i+2], 16) / 255
        for i in (0, 2, 4)
    )


base_rgb = np.array(
    hex_to_rgb(COL)
)


def build_colors(pulse):

    zn = (
        Z - Z.min()
    ) / (
        Z.max() - Z.min() + 1e-9
    )

    brightness = 0.18 + 0.95 * zn
    brightness *= (
        0.74 + 0.26 * pulse
    )

    colors = np.zeros((*Z.shape, 4))

    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(
        colors,
        0,
        1,
    )


def draw_grid(pulse):

    glow_alpha = (
        GRID_GLOW_ALPHA
        * (0.8 + 0.2 * pulse)
    )

    core_alpha = (
        GRID_CORE_ALPHA
        * (0.85 + 0.15 * pulse)
    )

    for i in range(
        0,
        X.shape[0],
        GRID_STEP_U,
    ):

        ax.plot(
            X[i],
            Y[i],
            Z[i],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[i],
            Y[i],
            Z[i],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []

tau = 2 * np.pi

for frame_idx in range(FRAMES):

    t = frame_idx / FRAMES

    pulse = (
        0.5
        + 0.5 * np.sin(
            tau * 4 * t
        )
    )

    ax.clear()

    ax.set_facecolor(BG)

    setup_axes()

    ax.view_init(
        elev=24 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(pulse)

    fig.canvas.draw()

    width, height = (
        fig.canvas.get_width_height()
    )

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(
        height,
        width,
        4,
    )

    frames.append(
        frame.copy()
    )

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(out_file)

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

animations/Math/aizawa_attractor_tube_v1.webm


[out#0/webm @ 0x11e715e50] video:1452KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.128433%
frame=  192 fps= 50 q=32.0 Lsize=    1454KiB time=00:00:08.00 bitrate=1489.1kbits/s speed=2.08x    


# Thomas Attractor

In [19]:
# Thomas Attractor — rotating mathematical sculpture
# Output: media-site/animations/Math/thomas_attractor_v1.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "thomas_attractor_v1"


# -----------------------------------------------------------------------------
# Thomas attractor
#
# dx/dt = sin(y) - b x
# dy/dt = sin(z) - b y
# dz/dt = sin(x) - b z
# -----------------------------------------------------------------------------

B = 0.19
DT = 0.015
STEPS = 52000
SKIP = 4000
DECIMATE = 2

x = np.zeros(STEPS)
y = np.zeros(STEPS)
z = np.zeros(STEPS)

x[0] = 0.1
y[0] = 0.0
z[0] = 0.0

for i in range(STEPS - 1):
    dx = np.sin(y[i]) - B * x[i]
    dy = np.sin(z[i]) - B * y[i]
    dz = np.sin(x[i]) - B * z[i]

    x[i + 1] = x[i] + DT * dx
    y[i + 1] = y[i] + DT * dy
    z[i + 1] = z[i] + DT * dz


X = x[SKIP::DECIMATE]
Y = y[SKIP::DECIMATE]
Z = z[SKIP::DECIMATE]

X -= X.mean()
Y -= Y.mean()
Z -= Z.mean()

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.8
Y = Y / scale * 4.8
Z = Z / scale * 4.8


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=26 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    # Glow layer
    ax.plot(
        X,
        Y,
        Z,
        color="#9ffcff",
        linewidth=5.0,
        alpha=0.06 + 0.05 * pulse,
    )

    # Core trajectory
    ax.plot(
        X,
        Y,
        Z,
        color=COL,
        linewidth=1.05,
        alpha=0.92,
    )

    # A few brighter orbit accents
    n = len(X)
    for offset in [0.0, 0.25, 0.5, 0.75]:
        idx = int((offset + t) % 1.0 * n)
        ax.scatter(
            [X[idx]],
            [Y[idx]],
            [Z[idx]],
            s=18,
            color="white",
            alpha=0.45 + 0.35 * pulse,
            depthshade=False,
        )

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/thomas_attractor_v1.webm
Frames: 192
Size: 2399.1 KB


[out#0/webm @ 0x122e242b0] video:2397KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.078663%
frame=  192 fps= 40 q=32.0 Lsize=    2399KiB time=00:00:08.00 bitrate=2456.7kbits/s speed=1.68x    


# Gyroid Surface v2

In [20]:
# Gyroid Surface v2 — rotating mathematical sculpture, museum mesh-grid style
# Output: media-site/animations/Math/gyroid_surface_v2_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from skimage import measure
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from vizlib.animation_export import export_animation


ANIMATION_NAME = "gyroid_surface_v2_grid"


# -----------------------------------------------------------------------------
# Gyroid surface geometry
#
# sin(x) cos(y) + sin(y) cos(z) + sin(z) cos(x) = 0
# -----------------------------------------------------------------------------

N = 46
L = 2.25 * np.pi

x = np.linspace(-L, L, N)
y = np.linspace(-L, L, N)
z = np.linspace(-L, L, N)

Xv, Yv, Zv = np.meshgrid(x, y, z, indexing="ij")

F = (
    np.sin(Xv) * np.cos(Yv)
    + np.sin(Yv) * np.cos(Zv)
    + np.sin(Zv) * np.cos(Xv)
)

verts, faces, normals, values = measure.marching_cubes(
    F,
    level=0.0,
    spacing=(
        (x.max() - x.min()) / (N - 1),
        (y.max() - y.min()) / (N - 1),
        (z.max() - z.min()) / (N - 1),
    ),
)

verts[:, 0] -= verts[:, 0].mean()
verts[:, 1] -= verts[:, 1].mean()
verts[:, 2] -= verts[:, 2].mean()

verts /= np.max(np.abs(verts)) / 4.15

mesh_faces = verts[faces]


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.25, 3.25)
    ax.set_ylim(-3.25, 3.25)
    ax.set_zlim(-3.25, 3.25)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def make_surface_layer(pulse: float) -> Poly3DCollection:
    alpha = SURFACE_ALPHA * (0.80 + 0.20 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(base_rgb[0], base_rgb[1], base_rgb[2], alpha),
        edgecolor=(0, 0, 0, 0),
        linewidth=0.0,
        alpha=alpha,
    )


def make_edge_glow_layer(pulse: float) -> Poly3DCollection:
    alpha = GRID_GLOW_ALPHA * 0.70 * (0.75 + 0.25 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(0, 0, 0, 0),
        edgecolor=(base_rgb[0], base_rgb[1], base_rgb[2], alpha),
        linewidth=0.38,
        alpha=1.0,
    )


def make_edge_core_layer(pulse: float) -> Poly3DCollection:
    alpha = GRID_CORE_ALPHA * 0.80 * (0.82 + 0.18 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(0, 0, 0, 0),
        edgecolor=(1.0, 1.0, 1.0, alpha),
        linewidth=0.07,
        alpha=1.0,
    )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=30 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.add_collection3d(make_surface_layer(pulse))
    ax.add_collection3d(make_edge_glow_layer(pulse))
    ax.add_collection3d(make_edge_core_layer(pulse))

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Faces: {len(faces)}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/gyroid_surface_v2_grid.webm
Faces: 46548
Frames: 192
Size: 14958.7 KB


[out#0/webm @ 0x12cf063c0] video:14957KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.013620%
frame=  192 fps= 11 q=32.0 Lsize=   14959KiB time=00:00:08.00 bitrate=15317.7kbits/s speed=0.471x    


# Neovius Surface

In [21]:
# Neovius Surface — rotating mathematical sculpture, museum mesh-grid style
# Output: media-site/animations/Math/neovius_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from skimage import measure
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from vizlib.animation_export import export_animation


ANIMATION_NAME = "neovius_surface_v1_grid"


N = 46
L = 2.25 * np.pi

x = np.linspace(-L, L, N)
y = np.linspace(-L, L, N)
z = np.linspace(-L, L, N)

Xv, Yv, Zv = np.meshgrid(x, y, z, indexing="ij")

F = (
    3.0 * (np.cos(Xv) + np.cos(Yv) + np.cos(Zv))
    + 4.0 * np.cos(Xv) * np.cos(Yv) * np.cos(Zv)
)

verts, faces, normals, values = measure.marching_cubes(
    F,
    level=0.0,
    spacing=(
        (x.max() - x.min()) / (N - 1),
        (y.max() - y.min()) / (N - 1),
        (z.max() - z.min()) / (N - 1),
    ),
)

verts[:, 0] -= verts[:, 0].mean()
verts[:, 1] -= verts[:, 1].mean()
verts[:, 2] -= verts[:, 2].mean()

verts /= np.max(np.abs(verts)) / 4.15
mesh_faces = verts[faces]


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.25, 3.25)
    ax.set_ylim(-3.25, 3.25)
    ax.set_zlim(-3.25, 3.25)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def make_surface_layer(pulse: float) -> Poly3DCollection:
    alpha = SURFACE_ALPHA * (0.80 + 0.20 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(base_rgb[0], base_rgb[1], base_rgb[2], alpha),
        edgecolor=(0, 0, 0, 0),
        linewidth=0.0,
        alpha=alpha,
    )


def make_edge_glow_layer(pulse: float) -> Poly3DCollection:
    alpha = GRID_GLOW_ALPHA * 0.70 * (0.75 + 0.25 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(0, 0, 0, 0),
        edgecolor=(base_rgb[0], base_rgb[1], base_rgb[2], alpha),
        linewidth=0.38,
        alpha=1.0,
    )


def make_edge_core_layer(pulse: float) -> Poly3DCollection:
    alpha = GRID_CORE_ALPHA * 0.80 * (0.82 + 0.18 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(0, 0, 0, 0),
        edgecolor=(1.0, 1.0, 1.0, alpha),
        linewidth=0.07,
        alpha=1.0,
    )


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=30 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.add_collection3d(make_surface_layer(pulse))
    ax.add_collection3d(make_edge_glow_layer(pulse))
    ax.add_collection3d(make_edge_core_layer(pulse))

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Faces: {len(faces)}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/neovius_surface_v1_grid.webm
Faces: 45928
Frames: 192
Size: 12046.9 KB



Формула:

$$ 3(\cos x+\cos y+\cos z)+4\cos x\cos y\cos z=0 $$

# Hyperbolic Coral

In [23]:
# Hyperbolic Coral — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/hyperbolic_coral_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "hyperbolic_coral_v1_grid"


# -----------------------------------------------------------------------------
# Hyperbolic coral geometry
# -----------------------------------------------------------------------------

u = np.linspace(0.0, 5.2 * np.pi, 260)
v = np.linspace(-0.75, 0.75, 130)

U, V = np.meshgrid(u, v)

growth = np.exp(0.075 * U)

radius = 0.23 * growth
tube = 1.0 + 0.35 * np.cos(6 * V)

X = radius * tube * np.cos(U)
Y = radius * tube * np.sin(U)
Z = 0.20 * U + 0.55 * radius * np.sin(V)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 5.0
Y = Y / scale * 5.0
Z = Z / scale * 5.0


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :],
                color=COL_GRID_U, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j],
                color=COL_GRID_V, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/hyperbolic_coral_v1_grid.webm
Frames: 192
Size: 609.4 KB


[out#0/webm @ 0x13e105ad0] video:608KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.303943%
frame=  192 fps= 65 q=32.0 Lsize=     609KiB time=00:00:08.00 bitrate= 624.0kbits/s speed=2.69x    


# Möbius Flower

In [24]:
# Möbius Flower — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/mobius_flower_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "mobius_flower_v1_grid"


# -----------------------------------------------------------------------------
# Möbius Flower geometry
#
# Möbius strip with radial petal modulation.
# -----------------------------------------------------------------------------

u = np.linspace(0.0, 2 * np.pi, 280)
v = np.linspace(-0.75, 0.75, 110)

U, V = np.meshgrid(u, v)

R0 = 2.1
PETALS = 8
A = 0.22

flower = 1.0 + A * np.cos(PETALS * U)

X = (R0 * flower + V * np.cos(U / 2.0)) * np.cos(U)
Y = (R0 * flower + V * np.cos(U / 2.0)) * np.sin(U)
Z = V * np.sin(U / 2.0)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.1
Y = Y / scale * 4.1
Z = Z / scale * 4.1


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)
    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    fn = (flower - flower.min()) / (flower.max() - flower.min() + 1e-9)

    brightness = 0.16 + 0.95 * fn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*flower.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :],
                color=COL_GRID_U, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j],
                color=COL_GRID_V, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/mobius_flower_v1_grid.webm
Frames: 192
Size: 1791.5 KB


[out#0/webm @ 0x1587083e0] video:1790KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.104826%
frame=  192 fps= 54 q=32.0 Lsize=    1791KiB time=00:00:08.00 bitrate=1834.5kbits/s speed=2.26x    


# Hopf Ring Field

In [25]:
# Hopf Ring Field — rotating mathematical sculpture
# Output: media-site/animations/Math/hopf_ring_field_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "hopf_ring_field_v1"


# -----------------------------------------------------------------------------
# Hopf circles
# -----------------------------------------------------------------------------

def hopf_circle(eta, n=500):
    t = np.linspace(0, 2*np.pi, n)

    a = np.cos(eta)
    b = np.sin(eta)

    denom = 1.0 + a * np.cos(t)

    x = b * np.cos(t) / denom
    y = b * np.sin(t) / denom
    z = a / denom

    return x, y, z


etas = np.linspace(
    -1.15,
    1.15,
    14,
)

circles = []

for eta in etas:

    x, y, z = hopf_circle(eta)

    circles.append(
        np.vstack([x, y, z]).T
    )


# -----------------------------------------------------------------------------
# Scale
# -----------------------------------------------------------------------------

all_pts = np.concatenate(
    circles,
    axis=0,
)

scale = np.max(
    np.abs(all_pts)
)

scaled = []

for C in circles:

    C = C / scale * 4.3

    scaled.append(C)


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(
    figsize=(W, H),
    dpi=DPI,
)

fig.patch.set_facecolor(BG)

ax = fig.add_subplot(
    111,
    projection="3d",
)

ax.set_facecolor(BG)


def setup_axes():

    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))

    ax.grid(False)
    ax.set_axis_off()


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []

tau = 2*np.pi

for frame_idx in range(FRAMES):

    t = frame_idx / FRAMES

    pulse = (
        0.5
        + 0.5*np.sin(
            tau * 4 * t
        )
    )

    ax.clear()

    ax.set_facecolor(BG)

    setup_axes()

    ax.view_init(
        elev=25 + 5*np.sin(tau*t),
        azim=360*t,
    )

    for k, C in enumerate(scaled):

        phase = (
            k / len(scaled)
        )

        alpha = (
            0.22
            + 0.35
            * np.sin(
                tau*(phase+t)
            )**2
        )

        # glow

        ax.plot(
            C[:,0],
            C[:,1],
            C[:,2],
            color="#9ffcff",
            linewidth=4.0,
            alpha=0.05 + alpha*0.10,
        )

        # core

        ax.plot(
            C[:,0],
            C[:,1],
            C[:,2],
            color=COL,
            linewidth=0.95,
            alpha=0.55 + alpha*0.35,
        )

    fig.canvas.draw()

    width, height = (
        fig.canvas.get_width_height()
    )

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(
        height,
        width,
        4,
    )

    frames.append(
        frame.copy()
    )

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(out_file)

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

animations/Math/hopf_ring_field_v1.webm


[out#0/webm @ 0x126a04080] video:48KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 3.712382%
frame=  192 fps= 99 q=32.0 Lsize=      50KiB time=00:00:08.00 bitrate=  50.9kbits/s speed=4.14x    
